# Fundus classification trên Kaggle

Notebook này chứa trực tiếp toàn bộ code training cho Swin-T, EfficientNetV2-M và weighted soft-voting ensemble.

Cần attach dataset `fundus_224_pad_v1` vào Notebook và bật `Settings → Accelerator → GPU`. Nếu dùng pretrained ImageNet weights, bật thêm Internet.

In [ ]:
from pathlib import Path
import subprocess

# Không bắt buộc cho training vì notebook đã chứa đầy đủ code.
# Điền URL thật nếu muốn clone repository để lưu phiên bản source.
REPO_URL = "https://github.com/nguyenquananh-2212/diabetic-retinopathy.git"
REPO_DIR = Path("/kaggle/working/Fundus_Project")

if not REPO_DIR.exists() and "YOUR_USERNAME" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    print("Repository cloned:", REPO_DIR)
elif REPO_DIR.exists():
    print("Repository already exists:", REPO_DIR)
else:
    print("Chưa clone repository. Hãy thay YOUR_USERNAME nếu cần clone GitHub.")

In [ ]:
import csv
import gc
import json
import random
import shutil
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import (
    EfficientNet_V2_M_Weights,
    Swin_T_Weights,
    efficientnet_v2_m,
    swin_t,
)
from tqdm.auto import tqdm

DATA_ROOT = Path("/kaggle/input/fundus-processed/fundus_224_pad_v1")
if not ((DATA_ROOT / "train").is_dir() and (DATA_ROOT / "val").is_dir()):
    candidates = []
    for dataset_dir in Path("/kaggle/input").iterdir():
        if not dataset_dir.is_dir():
            continue
        for candidate in (dataset_dir, dataset_dir / "fundus_224_pad_v1"):
            if (candidate / "train").is_dir() and (candidate / "val").is_dir():
                candidates.append(candidate)
    if len(candidates) != 1:
        raise FileNotFoundError("Không tìm thấy duy nhất dataset fundus_224_pad_v1. Hãy sửa DATA_ROOT.")
    DATA_ROOT = candidates[0]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_CLASSES = 5
IMAGE_SIZE = 224
NUM_WORKERS = 2
SEED = 42
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("DATA_ROOT:", DATA_ROOT)

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_dataset = ImageFolder(DATA_ROOT / "train", transform=train_transform)
val_dataset = ImageFolder(DATA_ROOT / "val", transform=val_transform)
test_dataset = ImageFolder(DATA_ROOT / "test", transform=val_transform)
assert train_dataset.classes == val_dataset.classes == test_dataset.classes
assert len(train_dataset.classes) == NUM_CLASSES

def make_loader(dataset, batch_size, shuffle):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )

print("Classes:", train_dataset.classes)
print("Train/Val/Test:", len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
def build_swin(pretrained=True):
    weights = Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
    model = swin_t(weights=weights)
    model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    return model

def build_efficientnet(pretrained=True):
    weights = EfficientNet_V2_M_Weights.IMAGENET1K_V1 if pretrained else None
    model = efficientnet_v2_m(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    return model

def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch, total_epochs):
    model.train()
    total_loss = 0.0
    total_items = 0
    progress = tqdm(loader, desc=f"Train {epoch}/{total_epochs}", dynamic_ncols=True)
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)
        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        count = labels.size(0)
        total_loss += loss.item() * count
        total_items += count
        progress.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / max(total_items, 1)

@torch.no_grad()
def evaluate(model, loader, criterion, epoch, total_epochs):
    model.eval()
    total_loss = 0.0
    total_items = 0
    targets, predictions = [], []
    progress = tqdm(loader, desc=f"Val   {epoch}/{total_epochs}", dynamic_ncols=True)
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)
        count = labels.size(0)
        total_loss += loss.item() * count
        total_items += count
        targets.extend(labels.cpu().tolist())
        predictions.extend(logits.argmax(1).cpu().tolist())
        progress.set_postfix(loss=f"{loss.item():.4f}")
    return {
        "loss": total_loss / max(total_items, 1),
        "accuracy": accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, average="macro", zero_division=0),
        "qwk": cohen_kappa_score(targets, predictions, weights="quadratic"),
    }

def train_model(model, model_name, batch_size, epochs=20, patience=5):
    train_loader = make_loader(train_dataset, batch_size, True)
    val_loader = make_loader(val_dataset, batch_size, False)
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    output_dir = Path(f"/kaggle/working/checkpoints/{model_name}")
    output_dir.mkdir(parents=True, exist_ok=True)
    best_path = output_dir / "best.pt"
    best_qwk = float("-inf")
    no_improvement = 0
    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch, epochs)
        metrics = evaluate(model, val_loader, criterion, epoch, epochs)
        scheduler.step()
        record = {"epoch": epoch, "train_loss": train_loss, **metrics}
        history.append(record)
        print(record)
        checkpoint = {"model_state_dict": model.state_dict(), "model_name": model_name, "metrics": metrics, "classes": train_dataset.classes}
        torch.save(checkpoint, output_dir / "last.pt")
        if metrics["qwk"] > best_qwk:
            best_qwk = metrics["qwk"]
            no_improvement = 0
            torch.save(checkpoint, best_path)
            print("Saved:", best_path)
        else:
            no_improvement += 1
        if no_improvement >= patience:
            print("Early stopping")
            break
    (output_dir / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    return model, best_path

In [ ]:
# Train Swin-T
TRAIN_SWIN = True
if TRAIN_SWIN:
    swin_model, SWIN_BEST = train_model(build_swin(pretrained=True), "swin_t", batch_size=4)
else:
    SWIN_BEST = Path("/kaggle/working/checkpoints/swin_t/best.pt")
print("Swin checkpoint:", SWIN_BEST)

In [ ]:
# Train EfficientNetV2-M
TRAIN_EFFICIENTNET = True
if TRAIN_EFFICIENTNET:
    efficientnet_model, EFFICIENTNET_BEST = train_model(build_efficientnet(pretrained=True), "efficientnet_v2_m", batch_size=4)
else:
    EFFICIENTNET_BEST = Path("/kaggle/working/checkpoints/efficientnet_v2_m/best.pt")
print("EfficientNet checkpoint:", EFFICIENTNET_BEST)

In [ ]:
def load_checkpoint(builder, path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    model = builder(pretrained=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model.to(DEVICE).eval()

@torch.no_grad()
def predict_probabilities(model, name):
    loader = make_loader(test_dataset, batch_size=4, shuffle=False)
    probabilities = []
    for images, _ in tqdm(loader, desc=f"Predict {name}", dynamic_ncols=True):
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            probabilities.append(torch.softmax(model(images), dim=1).float().cpu())
    return torch.cat(probabilities)

swin_model = load_checkpoint(build_swin, SWIN_BEST)
swin_probabilities = predict_probabilities(swin_model, "swin_t")
del swin_model
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

efficientnet_model = load_checkpoint(build_efficientnet, EFFICIENTNET_BEST)
efficientnet_probabilities = predict_probabilities(efficientnet_model, "efficientnet_v2_m")
del efficientnet_model
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Weighted soft voting và lưu kết quả ensemble
SWIN_WEIGHT = 0.5
EFFICIENTNET_WEIGHT = 0.5
ensemble_probabilities = (SWIN_WEIGHT * swin_probabilities + EFFICIENTNET_WEIGHT * efficientnet_probabilities) / (SWIN_WEIGHT + EFFICIENTNET_WEIGHT)
predictions = ensemble_probabilities.argmax(dim=1)
targets = [label for _, label in test_dataset.samples]

metrics = {
    "accuracy": float(accuracy_score(targets, predictions.tolist())),
    "macro_f1": float(f1_score(targets, predictions.tolist(), average="macro", zero_division=0)),
    "qwk": float(cohen_kappa_score(targets, predictions.tolist(), weights="quadratic")),
    "swin_weight": SWIN_WEIGHT,
    "efficientnet_weight": EFFICIENTNET_WEIGHT,
}

RESULT_DIR = Path("/kaggle/working/results/ensemble_swin_efficientnet_v2_m")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

with (RESULT_DIR / "predictions.csv").open("w", newline="", encoding="utf-8-sig") as output_file:
    fields = ["path", "true_label", "predicted_label"] + [f"probability_{i}" for i in range(NUM_CLASSES)]
    writer = csv.DictWriter(output_file, fieldnames=fields)
    writer.writeheader()
    for index, (path, true_label) in enumerate(test_dataset.samples):
        row = {"path": path, "true_label": true_label, "predicted_label": int(predictions[index])}
        row.update({f"probability_{i}": float(ensemble_probabilities[index, i]) for i in range(NUM_CLASSES)})
        writer.writerow(row)

print(json.dumps(metrics, indent=2))
archive = shutil.make_archive("/kaggle/working/fundus_results", "zip", Path("/kaggle/working/results"))
print("Saved:", RESULT_DIR)
print("Archive:", archive)